# Feature Engineering — кейс № 6

Цель работы — подготовить признаки для прогноза средней активной мощности металлургического завода на следующие 15 минут.

В ноутбуке создаются 10 признаков: 4 лаговых, 2 скользящих средних и 4 календарных.

## 1. Загрузка и первичный осмотр данных

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/red77789/team6/main/Steel_industry_data.csv'

df = pd.read_csv(DATA_URL)
df.head()

In [ ]:
print(f'Размер исходной таблицы: {df.shape}')
display(df.info())
display(df.isna().sum())

## 2. Подготовка временного столбца

Сортировка по времени обязательна: лаги и скользящие средние должны использовать только прошлые наблюдения.

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

intervals = df['date'].diff().dropna()
print('Начало данных:', df['date'].min())
print('Конец данных:', df['date'].max())
print('Основной интервал измерений:', intervals.mode().iloc[0])
print('Количество пропусков по времени:', (intervals != pd.Timedelta(minutes=15)).sum())

## 3. Лаговые признаки

Лаг — это значение энергопотребления в прошлом. Например, lag_1 — потребление 15 минут назад, а lag_96 — в то же время сутки назад.

In [ ]:
df['lag_1'] = df['Usage_kWh'].shift(1)
df['lag_4'] = df['Usage_kWh'].shift(4)
df['lag_96'] = df['Usage_kWh'].shift(96)
df['lag_672'] = df['Usage_kWh'].shift(672)

## 4. Скользящие средние

Средние рассчитываются только по предыдущим значениям. Поэтому модель не получает информацию о потреблении в момент, который должна предсказать.

In [ ]:
past_usage = df['Usage_kWh'].shift(1)
df['rolling_mean_4'] = past_usage.rolling(window=4).mean()
df['rolling_mean_24'] = past_usage.rolling(window=24).mean()

## 5. Календарные признаки

Время заранее известно, поэтому час, день недели, выходной и месяц можно использовать для прогноза.

In [ ]:
df['hour'] = df['date'].dt.hour
df['day_of_week'] = df['date'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
df['month'] = df['date'].dt.month

## 6. Целевая переменная и итоговая таблица

Исходный показатель Usage_kWh измерен за 15 минут. Средняя активная мощность в кВт равна потреблённой энергии, умноженной на 4.

Текущее значение Usage_kWh не включаем в признаки: иначе модель увидела бы ответ заранее.

In [ ]:
df['target_power_kw'] = df['Usage_kWh'] * 4

feature_columns = [
    'lag_1', 'lag_4', 'lag_96', 'lag_672',
    'rolling_mean_4', 'rolling_mean_24',
    'hour', 'day_of_week', 'is_weekend', 'month'
]

df_final = df[['date', *feature_columns, 'target_power_kw']].dropna().copy()

print(f'Размер итоговой таблицы: {df_final.shape}')
print(f'Пропусков в итоговой таблице: {df_final.isna().sum().sum()}')
df_final.head()

## 7. Результат

Получена таблица из 34 368 наблюдений: дата, 10 признаков и целевая переменная target_power_kw.

Для корректной проверки модели данные нужно разделять по времени: ранние даты использовать для обучения, поздние — для проверки. Полиномиальные признаки и регуляризацию следует проверять в отдельном ноутбуке с моделью.

In [ ]:
OUTPUT_FILE = 'steel_features_power_team6.csv'
df_final.to_csv(OUTPUT_FILE, index=False)
print(f'Файл сохранён: {OUTPUT_FILE}')

# Для Google Colab можно раскомментировать две строки ниже:
# from google.colab import files
# files.download(OUTPUT_FILE)